# Part 2: Add Fabric IQ (Ontology) as a knowledge source

This notebook creates a **`fabricIQ`** knowledge source backed by a real Fabric
ontology and runs a grounded retrieve call through the KB.

### Prerequisites

| Item | Details |
|------|---------|
| `.env` file | In the `notebooks/` directory (same folder as this notebook) — contains `BIGBUGS_ENDPOINT`, `BIGBUGS_API_KEY`. |
| Azure CLI | Logged in (`az login`) to a principal with Fabric access. |
| Fabric workspace | Workspace `b0a49ce1-0af9-4d14-a6b6-f6509b0aeec8` with ontology `cee04244-7307-4266-b760-31c372f98550`. |

> ⚠️ **The `.env` is currently in testing state — ask Matt for setup credentials.**
> Never check secrets into this repo.

### What works on the current BigBugs stamp

- `fabricIQ` knowledge source CRUD ✅
- KB create + retrieve with Fabric token ✅
- `fabricEndpoint` must be `https://dxtapi.fabric.microsoft.com` — wrong values
  fail at retrieve time with misleading errors.

## 1 — Setup

In [ ]:
import json, os, subprocess, shutil, time
from pathlib import Path
from datetime import datetime, timezone

import requests

ENV_PATH = Path(".").resolve() / ".env"

def load_env(path: Path) -> dict:
    values = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        values[k.strip()] = v.strip()
    return values

env = load_env(ENV_PATH)
ENDPOINT = env["BIGBUGS_ENDPOINT"].rstrip("/")
API_KEY  = env["BIGBUGS_API_KEY"]
API_VERSION = "2026-05-01-preview"

# Fabric IQ identifiers (from the bug-bash setup)
WORKSPACE_ID = "b0a49ce1-0af9-4d14-a6b6-f6509b0aeec8"
ONTOLOGY_ID  = "cee04244-7307-4266-b760-31c372f98550"
FABRIC_ENDPOINT = "https://dxtapi.fabric.microsoft.com"

session = requests.Session()
session.headers.update({"api-key": API_KEY, "Accept": "application/json"})

def url(path: str) -> str:
    sep = "&" if "?" in path else "?"
    return f"{ENDPOINT}{path}{sep}api-version={API_VERSION}"

def pp(r: requests.Response):
    print(f"{r.status_code} {r.reason}")
    try:
        print(json.dumps(r.json(), indent=2))
    except Exception:
        print(r.text[:500])

stamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
KS_NAME = f"lab532-fabriciq-{stamp}"
KB_NAME = f"lab532-fabriciq-kb-{stamp}"

print(f"Endpoint  : {ENDPOINT}")
print(f"Workspace : {WORKSPACE_ID}")
print(f"Ontology  : {ONTOLOGY_ID}")
print(f"KS        : {KS_NAME}")
print(f"KB        : {KB_NAME}")

## 2 — Obtain a Fabric access token

The `fabricIQ` retrieve call requires a Fabric access token passed as
`x-ms-query-source-authorization: accessToken-<token>`.

We get this from `az account get-access-token --resource https://api.fabric.microsoft.com`.

In [ ]:
az_exe = shutil.which("az.cmd") or shutil.which("az")
if az_exe is None:
    raise FileNotFoundError("Azure CLI not found — run `az login` first.")

token_result = subprocess.run(
    [az_exe, "account", "get-access-token",
     "--resource", "https://api.fabric.microsoft.com",
     "--output", "json"],
    check=True, capture_output=True, text=True,
)
FABRIC_TOKEN = json.loads(token_result.stdout)["accessToken"]
QUERY_SOURCE_AUTH = f"accessToken-{FABRIC_TOKEN}"
print(f"Fabric token obtained ({len(FABRIC_TOKEN)} chars)")

## 3 — Create the `fabricIQ` knowledge source

In [ ]:
ks_body = {
    "name": KS_NAME,
    "kind": "fabricIQ",
    "description": "Lab 532 Fabric IQ ontology source",
    "fabricIQParameters": {
        "workspaceId": WORKSPACE_ID,
        "ontologyId": ONTOLOGY_ID,
        "fabricEndpoint": FABRIC_ENDPOINT,
    },
}

r = session.put(url(f"/knowledgesources('{KS_NAME}')"), json=ks_body,
                headers={"Prefer": "return=representation"})
pp(r)

## 4 — Create a KB referencing the Fabric IQ source

In [ ]:
kb_body = {
    "name": KB_NAME,
    "description": "Lab 532 KB — Fabric IQ ontology",
    "outputMode": "extractiveData",
    "retrievalReasoningEffort": {"kind": "minimal"},
    "knowledgeSources": [{"name": KS_NAME}],
}

r = session.put(url(f"/knowledgebases('{KB_NAME}')"), json=kb_body,
                headers={"Prefer": "return=representation"})
pp(r)

## 5 — Retrieve: ask a Fabric Ontology question

The ontology contains airline/supply-chain data with Customer, Passenger, and related entities.
Queries must match the ontology schema — unrecognized entity types will fail.

In [ ]:
retrieve_body = {
    "includeActivity": True,
    "intents": [
        {"type": "semantic",
         "search": "Show all Customer entities with LoyaltyTier 'gold' and IsActiveCustomer set to True."}
    ],
    "knowledgeSourceParams": [
        {
            "kind": "fabricIQ",
            "knowledgeSourceName": KS_NAME,
            "includeReferenceSourceData": True,
            "includeReferences": True,
        }
    ],
}

r = session.post(
    url(f"/knowledgebases('{KB_NAME}')/retrieve"),
    json=retrieve_body,
    headers={"x-ms-query-source-authorization": QUERY_SOURCE_AUTH},
    timeout=60,
)
pp(r)
if r.status_code == 200:
    print("\n✅ Fabric IQ retrieve succeeded!")
else:
    print(f"\n⚠️ Fabric IQ retrieve returned {r.status_code} — this may be a downstream Fabric service issue.")
    print("   The NL-to-ontology translation can fail for queries that don't match the ontology schema.")


### Try another question

In [ ]:
retrieve_body_2 = {
    "includeActivity": True,
    "intents": [
        {"type": "semantic",
         "search": "Which customers have LoyaltyTier 'silver'? Show their names."}
    ],
    "knowledgeSourceParams": [
        {
            "kind": "fabricIQ",
            "knowledgeSourceName": KS_NAME,
            "includeReferenceSourceData": True,
            "includeReferences": True,
        }
    ],
}

r = session.post(
    url(f"/knowledgebases('{KB_NAME}')/retrieve"),
    json=retrieve_body_2,
    headers={"x-ms-query-source-authorization": QUERY_SOURCE_AUTH},
    timeout=60,
)
pp(r)
if r.status_code == 200:
    print("\n✅ Fabric IQ retrieve succeeded!")
else:
    print(f"\n⚠️ Fabric IQ retrieve returned {r.status_code} — this may be a downstream Fabric service issue.")
    print("   The NL-to-ontology translation can fail for queries that don't match the ontology schema.")


## 6 — Cleanup

In [ ]:
for label, path in [
    ("KB", f"/knowledgebases('{KB_NAME}')"),
    ("KS", f"/knowledgesources('{KS_NAME}')"),
]:
    r = session.delete(url(path))
    print(f"Delete {label}: {r.status_code} {r.reason}")

## Known issues

| Issue | Detail |
|-------|--------|
| `fabricEndpoint` validation | Wrong endpoints are accepted at KS create but fail at retrieve time with misleading 404/405 errors. |
| Token lifetime | Fabric tokens expire quickly — re-run the token cell if you get 401s. |

➡️ Continue to [Part 3: Add Work IQ to the KB](part3-work-iq-to-kb.ipynb).| NL query translation | Retrieve can fail with 502 if the natural-language query doesn't map to the ontology schema. Try rephrasing. |
